# 03. 추론·배포 템플릿 — TensorRT · ONNX Runtime · vLLM
1. PyTorch 즉시 실행 vs **Torch-TensorRT** 컴파일 비교 (비전 모델)
2. **ONNX → ONNX Runtime (TensorRT EP)** 표준 배포 경로
3. **vLLM OpenAI 호환 API** 호출 + JSON 구조화 출력 (서버는 호스트에서 `optimize/30_serving.sh vllm-up`)

In [ ]:
import time, torch, torchvision
DEV = "cuda"
model = torchvision.models.resnet50(weights=None).eval().to(DEV).half()
x = torch.rand(32, 3, 224, 224, device=DEV).half()
def bench(fn, n=50):
    for _ in range(10): fn()
    torch.cuda.synchronize(); t = time.time()
    for _ in range(n): fn()
    torch.cuda.synchronize(); return n * x.shape[0] / (time.time() - t)
with torch.no_grad():
    eager = bench(lambda: model(x))
print(f"PyTorch FP16 eager: {eager:8.1f} img/s")

In [ ]:
import torch_tensorrt
trt_model = torch_tensorrt.compile(model, ir="dynamo", inputs=[x], enabled_precisions={torch.half})
with torch.no_grad():
    trt = bench(lambda: trt_model(x))
print(f"Torch-TensorRT FP16: {trt:8.1f} img/s  ({trt/eager:.2f}x)")

In [ ]:
import onnxruntime as ort, numpy as np, os
path = "/tmp/resnet50.onnx"
torch.onnx.export(model.float(), x.float()[:1], path, opset_version=17,
                  input_names=["input"], output_names=["logits"], dynamic_axes={"input": {0: "batch"}})
os.makedirs("/tmp/trt_cache", exist_ok=True)
providers = [("TensorrtExecutionProvider", {"trt_fp16_enable": True, "trt_engine_cache_enable": True,
                                            "trt_engine_cache_path": "/tmp/trt_cache"}),
             "CUDAExecutionProvider", "CPUExecutionProvider"]
sess = ort.InferenceSession(path, providers=providers)
print("실제 사용 중인 실행 공급자:", sess.get_providers())
xn = x.float().cpu().numpy()
t = time.time(); sess.run(None, {"input": xn}); print(f"첫 실행(엔진 빌드 포함) {time.time()-t:.1f}s")
t = time.time()
for _ in range(20): sess.run(None, {"input": xn})
print(f"ONNX Runtime: {20*xn.shape[0]/(time.time()-t):8.1f} img/s")
model.half()

In [ ]:
# vLLM (호스트에서 `sudo bash optimize/30_serving.sh vllm-up` 으로 먼저 기동)
import requests, json, os, time
VLLM_URL = os.environ.get("VLLM_URL", "http://172.17.0.1:8000/v1")   # 도커 브리지에서 본 호스트
try:
    model_id = requests.get(f"{VLLM_URL}/models", timeout=3).json()["data"][0]["id"]
except Exception as e:
    model_id = None; print("vLLM 서버 미기동:", e)
if model_id:
    schema = {"type": "object", "properties": {
                "공종": {"type": "string"}, "자재": {"type": "array", "items": {"type": "string"}},
                "예상일수": {"type": "integer"}}, "required": ["공종", "자재", "예상일수"]}
    body = {"model": model_id, "max_tokens": 256, "temperature": 0,
            "messages": [{"role": "user", "content": "욕실 타일 교체 공사를 JSON 으로 요약해줘."}],
            "guided_json": schema}                      # 구조화 출력 (스키마 강제)
    t = time.time(); r = requests.post(f"{VLLM_URL}/chat/completions", json=body, timeout=120).json()
    dt = time.time() - t; out = r["choices"][0]["message"]["content"]
    print(json.dumps(json.loads(out), ensure_ascii=False, indent=2))
    print(f"{r['usage']['completion_tokens']/dt:.1f} tok/s (전체 응답 기준)")